Aim: extract geospatial data from Open Street Map (OSM) files in downloaded from Geofabrik website in PBF format. Currently sertup for working for lines and filtering by specific tags. Options to have all tags were problematic as part of the chain uses geodataframes where duplicate tags in different cases were converted to lower case causing errors from duplicate headings.

In [10]:
import osmium
import geopandas as gpd
from shapely.geometry import LineString
import os
import time

In [11]:
input_dir = r"C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\raw\roads\osm\osm_regional_250521"
# output_dir = r"C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\railways\osm\osm_regional_250521"

output_dir = r"C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521"

# ==== Example file selection logic ====
extract_types = ['waterways', 'boat_access', 'ferry_routes', 'railways'] # 'highways']


In [12]:

class GeometryExtractor(osmium.SimpleHandler):
    def __init__(self, extract_types):
        super().__init__()
        self.extract_types = extract_types
        self.highways = []
        self.railways = []
        self.waterways = []
        self.boat_access = []
        self.ferry_routes = []
        self.error_count = 0
        self.count = 0
        self.start_time = time.time()

    def way(self, w):
        self.count += 1
        if self.count % 10000 == 0:
            elapsed = time.time() - self.start_time
            print(f"Processed {self.count} ways in {elapsed:.1f}s")

        try:
            coords = [(n.lon, n.lat) for n in w.nodes if hasattr(n, 'lon') and hasattr(n, 'lat')]
            if len(coords) < 2:
                return
            line = LineString(coords)
            way_dict = {
                'id': w.id,
                'name': w.tags.get('name', ''),
                'geometry': line
            }

            if 'highways' in self.extract_types and 'highway' in w.tags:
                way_dict.update({
                    'highway': w.tags.get('highway', ''),
                    'surface': w.tags.get('surface', '')
                })
                self.highways.append(way_dict)

            if 'railways' in self.extract_types and 'railway' in w.tags:
                way_dict.update({
                    'railway': w.tags.get('railway', ''),
                    'tunnel': w.tags.get('tunnel', ''),
                    'bridge': w.tags.get('bridge', ''),
                    'electrified': w.tags.get('electrified', ''),
                    'usage': w.tags.get('usage', ''),
                    'service': w.tags.get('service', ''),
                    'layer': w.tags.get('layer', '')
                })
                self.railways.append(way_dict)

            if 'waterways' in self.extract_types and 'waterway' in w.tags:
                way_dict.update({
                    'waterway': w.tags.get('waterway', ''),
                    'boat': w.tags.get('boat', ''),
                    'motorboat': w.tags.get('motorboat', ''),
                    'canoe': w.tags.get('canoe', ''),
                    'kayak': w.tags.get('kayak', ''),
                    'sailing': w.tags.get('sailing', ''),
                    'navigable': w.tags.get('navigable', ''),
                    'navigation': w.tags.get('navigation', ''),
                    'navigation:system': w.tags.get('navigation:system', ''),
                    'usage': w.tags.get('usage', ''),
                    'cemt': w.tags.get('cemt', ''),
                    'maxdraft': w.tags.get('maxdraft', ''),
                    'maxlength': w.tags.get('maxlength', ''),
                    'maxwidth': w.tags.get('maxwidth', ''),
                    'maxheight': w.tags.get('maxheight', ''),
                    'width': w.tags.get('width', ''),
                    'depth': w.tags.get('depth', ''),
                    'mooring': w.tags.get('mooring', ''),
                    'lock': w.tags.get('lock', ''),
                    'bridge': w.tags.get('bridge', ''),
                    'tunnel': w.tags.get('tunnel', ''),
                    'layer': w.tags.get('layer', ''),
                    'barrier': w.tags.get('barrier', ''),
                    'route': w.tags.get('route', ''),
                    'designation': w.tags.get('designation', ''),
                    'class': w.tags.get('class', ''),
                    'ref': w.tags.get('ref', ''),
                    'operator': w.tags.get('operator', ''),
                    'admin_level': w.tags.get('admin_level', ''),
                    'intermittent': w.tags.get('intermittent', ''),
                    'seasonal': w.tags.get('seasonal', ''),
                    'ford': w.tags.get('ford', ''),
                    'seamark:type': w.tags.get('seamark:type', '')
                })
                self.waterways.append(way_dict)

            if 'boat_access' in self.extract_types and (
                w.tags.get('mooring') or
                w.tags.get('man_made') == 'pier' or
                w.tags.get('amenity') == 'boat_ramp' or
                w.tags.get('leisure') == 'marina' or
                w.tags.get('landing')
            ):
                way_dict.update({
                    'mooring': w.tags.get('mooring', ''),
                    'man_made': w.tags.get('man_made', ''),
                    'amenity': w.tags.get('amenity', ''),
                    'leisure': w.tags.get('leisure', ''),
                    'landing': w.tags.get('landing', ''),
                    'natural': w.tags.get('natural', ''),
                    'access': w.tags.get('access', ''),
                    'operator': w.tags.get('operator', ''),
                    'ref': w.tags.get('ref', '')
                })
                self.boat_access.append(way_dict)

            if 'ferry_routes' in self.extract_types and w.tags.get('route') == 'ferry':
                way_dict.update({
                    'route': w.tags.get('route', ''),
                    'operator': w.tags.get('operator', ''),
                    'duration': w.tags.get('duration', ''),
                    'frequency': w.tags.get('frequency', ''),
                    'access': w.tags.get('access', ''),
                    'layer': w.tags.get('layer', ''),
                    'ref': w.tags.get('ref', '')
                })
                self.ferry_routes.append(way_dict)

        except Exception as e:
            self.error_count += 1
            print(f"Error in way {w.id}: {e}")


In [13]:

os.makedirs(output_dir, exist_ok=True)

input_files = [f for f in os.listdir(input_dir) if f.endswith("-latest.osm.pbf")]
files_to_process = []

for file in input_files:
    country_name = file.replace("-latest.osm.pbf", "").replace("-", "_")
    input_path = os.path.join(input_dir, file)
    expected_outputs = [os.path.join(output_dir, f"{country_name}_{etype}.gpkg") for etype in extract_types]
    if all(os.path.exists(f) for f in expected_outputs):
        print(f"All outputs exist for {country_name}. Skipping.")
    else:
        files_to_process.append((input_path, country_name))

for input_path, country_name in files_to_process:
    print(f"Processing {country_name}...")
    reader = osmium.io.Reader(input_path)

   
    # Assign node locations to ways to avoid 'invalid location' errors
    idx = osmium.index.create_map("sparse_mem_array")
    location_handler = osmium.NodeLocationsForWays(idx)
    location_handler.ignore_errors()  # optional: skip missing nodes

    handler = GeometryExtractor(extract_types)
    osmium.apply(reader, location_handler, handler)
    reader.close()


    if 'highways' in extract_types and handler.highways:
        gdf = gpd.GeoDataFrame(handler.highways, crs="EPSG:4326")
        gdf.to_file(os.path.join(output_dir, f"{country_name}_highways.gpkg"))

    if 'railways' in extract_types and handler.railways:
        gdf = gpd.GeoDataFrame(handler.railways, crs="EPSG:4326")
        gdf.to_file(os.path.join(output_dir, f"{country_name}_railways.gpkg"))

    if 'waterways' in extract_types and handler.waterways:
        gdf = gpd.GeoDataFrame(handler.waterways, crs="EPSG:4326")
        gdf.to_file(os.path.join(output_dir, f"{country_name}_waterways.gpkg"))

    if 'boat_access' in extract_types and handler.boat_access:
        gdf = gpd.GeoDataFrame(handler.boat_access, crs="EPSG:4326")
        gdf.to_file(os.path.join(output_dir, f"{country_name}_boat_access.gpkg"))

    if 'ferry_routes' in extract_types and handler.ferry_routes:
        gdf = gpd.GeoDataFrame(handler.ferry_routes, crs="EPSG:4326")
        gdf.to_file(os.path.join(output_dir, f"{country_name}_ferry_routes.gpkg"))

    print(f"Errors encountered: {handler.error_count}")

Processing africa...
Processed 10000 ways in 151.0s
Processed 20000 ways in 153.1s
Processed 30000 ways in 157.8s
Processed 40000 ways in 164.1s
Processed 50000 ways in 167.4s
Processed 60000 ways in 172.4s
Processed 70000 ways in 174.8s
Processed 80000 ways in 176.8s
Processed 90000 ways in 179.0s
Processed 100000 ways in 180.6s
Processed 110000 ways in 182.6s
Processed 120000 ways in 184.7s
Processed 130000 ways in 187.0s
Processed 140000 ways in 188.9s
Processed 150000 ways in 190.4s
Processed 160000 ways in 192.8s
Processed 170000 ways in 194.1s
Processed 180000 ways in 196.6s
Processed 190000 ways in 199.5s
Processed 200000 ways in 202.4s
Processed 210000 ways in 204.4s
Processed 220000 ways in 205.8s
Processed 230000 ways in 208.6s
Processed 240000 ways in 210.2s
Processed 250000 ways in 212.2s
Processed 260000 ways in 214.6s
Processed 270000 ways in 217.6s
Processed 280000 ways in 218.9s
Processed 290000 ways in 220.8s
Processed 300000 ways in 222.2s
Processed 310000 ways in 223

MemoryError: bad allocation